In [ ]:
import pyemu
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

# ── Load PST files ──────────────────────────────────────────────────────────
m_d_prior    = 'pyemu/PMW-6-constant/master_pp_newweights_newinitialK'
m_d_noprior  = 'pyemu/PMW-6-constant/master_pp3_no_ss_no_prior'

pst_prior   = pyemu.Pst(os.path.join(m_d_prior,   'mf6.pst'))
pst_noprior = pyemu.Pst(os.path.join(m_d_noprior,  'mf6.pst'))

# ── Pull residuals and filter to non-zero weight obs only ───────────────────
def get_res(pst):
    res = pst.res.copy()
    res = res.loc[pst.nnz_obs_names]          # non-zero weight obs only
    res['obsgroup'] = pst.observation_data.loc[res.index, 'obgnme']
    return res

res_prior   = get_res(pst_prior)
res_noprior = get_res(pst_noprior)

# ── Layer mapping ────────────────────────────────────────────────────────────
layer_groups = {'obs1': 'Layer 1', 'obs2': 'Layer 2 (BPP)', 'obs3': 'Layer 3'}
layer_colors = {'obs1': '#4C72B0', 'obs2': '#DD8452', 'obs3': '#55A868'}

# ── Shared 1:1 line limits across all data ──────────────────────────────────
all_vals = np.concatenate([
    res_prior['measured'].values,   res_prior['modelled'].values,
    res_noprior['measured'].values, res_noprior['modelled'].values,
])
glob_min, glob_max = np.floor(all_vals.min()), np.ceil(all_vals.max())

# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 1 — 2×3 grid: rows = no-prior / prior, cols = layers
# ═══════════════════════════════════════════════════════════════════════════
fig1, axes1 = plt.subplots(2, 3, figsize=(15, 10), sharex=True, sharey=True)
fig1.suptitle('Obs vs. Sim — by layer', fontsize=16, y=1.01)

row_labels  = ['No prior', 'With prior']
row_data    = [res_noprior, res_prior]

for row_i, (label, res) in enumerate(zip(row_labels, row_data)):
    for col_i, (grp, grp_label) in enumerate(layer_groups.items()):
        ax = axes1[row_i, col_i]
        sub = res[res['obsgroup'] == grp]

        ax.scatter(sub['measured'], sub['modelled'],
                   color=layer_colors[grp], s=40, alpha=0.75, edgecolors='k', linewidths=0.4)

        ax.plot([glob_min, glob_max], [glob_min, glob_max],
                'k--', lw=1, label='1:1')

        r2 = np.corrcoef(sub['measured'], sub['modelled'])[0, 1] ** 2
        rmse = np.sqrt(np.mean((sub['measured'] - sub['modelled'])**2))
        ax.text(0.05, 0.92, f'R²={r2:.3f}\nRMSE={rmse:.4f} m',
                transform=ax.transAxes, fontsize=9,
                verticalalignment='top',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

        if row_i == 0:
            ax.set_title(grp_label, fontsize=13)
        if col_i == 0:
            ax.set_ylabel(f'{label}\nSimulated head (m)', fontsize=11)
        if row_i == 1:
            ax.set_xlabel('Observed head (m)', fontsize=11)

        ax.set_xlim(glob_min, glob_max)
        ax.set_ylim(glob_min, glob_max)
        ax.set_aspect('equal')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('obs_sim_by_layer_grid.png', dpi=150, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 2 — 2 panels: no-prior | prior, layers color-coded
# ═══════════════════════════════════════════════════════════════════════════
fig2, axes2 = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
fig2.suptitle('Obs vs. Sim — layers color-coded', fontsize=14)

for ax, res, title in zip(axes2, [res_noprior, res_prior], ['No prior', 'With prior']):
    for grp, grp_label in layer_groups.items():
        sub = res[res['obsgroup'] == grp]
        ax.scatter(sub['measured'], sub['modelled'],
                   label=grp_label, color=layer_colors[grp],
                   s=40, alpha=0.75, edgecolors='k', linewidths=0.4)

    ax.plot([glob_min, glob_max], [glob_min, glob_max], 'k--', lw=1)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Observed head (m)', fontsize=11)
    ax.set_xlim(glob_min, glob_max)
    ax.set_ylim(glob_min, glob_max)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9, loc='upper left')

axes2[0].set_ylabel('Simulated head (m)', fontsize=11)
plt.tight_layout()
plt.savefig('obs_sim_colorcoded_noprior_vs_prior.png', dpi=150, bbox_inches='tight')
plt.show()

# ═══════════════════════════════════════════════════════════════════════════
# FIGURE 3 — 3 panels: one per layer, prior vs. no-prior color-coded
# ═══════════════════════════════════════════════════════════════════════════
case_colors = {'No prior': '#E05C5C', 'With prior': '#5C7BE0'}

fig3, axes3 = plt.subplots(1, 3, figsize=(15, 5), sharex=True, sharey=True)
fig3.suptitle('Obs vs. Sim — prior vs. no-prior per layer', fontsize=14)

for ax, (grp, grp_label) in zip(axes3, layer_groups.items()):
    for res, case_label in [(res_noprior, 'No prior'), (res_prior, 'With prior')]:
        sub = res[res['obsgroup'] == grp]
        ax.scatter(sub['measured'], sub['modelled'],
                   label=case_label, color=case_colors[case_label],
                   s=40, alpha=0.75, edgecolors='k', linewidths=0.4)

    ax.plot([glob_min, glob_max], [glob_min, glob_max], 'k--', lw=1)
    ax.set_title(grp_label, fontsize=13)
    ax.set_xlabel('Observed head (m)', fontsize=11)
    ax.set_xlim(glob_min, glob_max)
    ax.set_ylim(glob_min, glob_max)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=9, loc='upper left')

axes3[0].set_ylabel('Simulated head (m)', fontsize=11)
plt.tight_layout()
plt.savefig('obs_sim_per_layer_prior_comparison.png', dpi=150, bbox_inches='tight')
plt.show()